<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/17-Using_LLMs_to_rank_chunks_as_the_Judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using LLMs to Rank Chunks as the Judge

This notebook demonstrates how to use a Large Language Model (LLM) as a judge to re-rank retrieved chunks in a RAG pipeline, improving retrieval quality before answer generation.

## What This Notebook Covers
- Using **RankGPT** for LLM-based re-ranking of retrieved nodes
- Building a **custom postprocessor** that uses an LLM to score and rank chunks
- Comparing retrieval results before and after LLM-based re-ranking

## Learning Objectives
By the end of this notebook, you will be able to:
1. Apply RankGPT as a post-retrieval re-ranking step in a LlamaIndex query engine
2. Implement a custom `BaseNodePostprocessor` that uses structured LLM output to score nodes
3. Understand how LLM-as-judge re-ranking improves RAG answer quality

## Prerequisites
- OpenAI API key
- Basic understanding of RAG pipelines and LlamaIndex
- Familiarity with Python classes and pydantic models

## Install Packages and Setup Variables


In [ ]:
!pip install -qU llama-index==0.14.19 openai==2.8.1 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 \
                 opentelemetry-api==1.38.0 llama-index-embeddings-openai==0.6.0 jedi==0.19.2 \
                 huggingface-hub==1.8.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 993.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "[OPENAI_API_KEY]"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")

# Load a Model


In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


Downloading vector store from Huggingface hub

In [ ]:
from huggingface_hub import hf_hub_download
vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip",repo_type="dataset",local_dir="/content")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [ ]:
!unzip /content/vectorstore.zip

Archive:  /content/vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


In [ ]:
# Load the vector store from the local storage.
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db2 = chromadb.PersistentClient(path="/content/ai_tutor_knowledge")
chroma_collection = db2.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding

# Create the index based on the vector store.
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=10)

res = query_engine.query("Explain how Advanced RAG works?")

In [ ]:
print(res.response)

Advanced RAG refers to a family of Retrieval-Augmented Generation techniques that go beyond the simple retrieve-then-generate pipeline by adding routing, evaluation, self-correction, and efficiency-focused drafting/verification steps. Key components and how they work together:

1. Multi-strategy retrieval and routing (Adaptive RAG)
- Route incoming queries to different retrievers or retrieval strategies depending on the question type (e.g., multi-hop vs. factual lookup, domain-specific vs. web search).
- Combine outputs from multiple retrieval perspectives (e.g., rewriting the query and retrieving for each rewrite) and fuse rankings (Reciprocal Rank Fusion) to produce a stronger, unified set of candidate documents.

2. Document quality checks and fallback (Corrective RAG)
- Evaluate retrieved documents for relevance before generation. If top retrieved docs are poor or off-domain, use a fallback retrieval (for example, a web search) to fetch better evidence.
- This lightweight retrieval

# RankGPT


In [ ]:
from llama_index.core.postprocessor.rankGPT_rerank import RankGPTRerank

rankGPT = RankGPTRerank(top_n=3,llm=Settings.llm, verbose=True)

In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
# The `node_postprocessors` function will be applied to the retrieved nodes.
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[rankGPT])

res = query_engine.query("Explain how Retrieval Augmented Generation (RAG) works?")

After Reranking, new rank list for nodes: [1, 3, 4, 0, 7, 9, 6, 8, 2, 5]

In [ ]:
print(res.response)

Retrieval-Augmented Generation (RAG) is a two-part approach that combines an information retrieval subsystem with a generative language model so the generator can ground its outputs on explicit external knowledge.

Main components

1. Retrieval
- Purpose: extract relevant documents or passages from an external corpus to supply factual context.
- Indexing: documents are organized for efficient lookup, e.g., via inverted indexes for sparse retrieval or dense vector encodings for dense retrieval.
- Searching: a retriever uses the index to fetch candidate passages for a user query. Retrieved results are often refined with rerankers.
- Non-parametric memory: the corpus (e.g., a dense vector index of Wikipedia) serves as an external, updatable store of knowledge.

2. Generation
- Purpose: produce coherent, contextually relevant responses using the query plus the retrieved content.
- Model: a seq2seq or large language model acts as the parametric memory and generation engine.
- Prompting and 

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text.strip())
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 1c324686-fad7-4a41-bfd3-d44f9612ca91
Title	 Evaluation of Retrieval-Augmented Generation: A Survey:Research Paper Information
Text	 Authors: Hao Yu, Aoran Gan, Kai Zhang, Shiwei Tong, Qi Liu, and Zhaofeng Liu.Numerous studies of Retrieval-Augmented Generation (RAG) systems have emerged from various perspectives since the advent of Large Language Models (LLMs). The RAG system comprises two primary components: Retrieval and Generation. The retrieval component aims to extract relevant information from various external knowledge sources. It involves two main phases, indexing and searching. Indexing organizes documents to facilitate efficient retrieval, using either inverted indexes for sparse retrieval or dense vector encoding for dense retrieval. The searching component utilizes these indexes to fetch relevant documents based on the user's query, often incorporating the optional rerankers to refine the ranking of the retrieved documents. The generation component utilizes the retr

# Custom Postprocessor


## The `Judger` Function


The following function will query GPT-4o mini to retrieve the top three nodes that have the highest similarity to the asked question.


In [ ]:
from pydantic import BaseModel
from llama_index.llms.openai import OpenAI
from llama_index.core.prompts import PromptTemplate


def judger(nodes, query):

    # The model's output template
    class OrderedNodes(BaseModel):
        """A node with the id and assigned score."""

        node_id: List[str]
        score: List[float]

    # Prepare the nodes and wrap them in <NODE></NODE> identifier, as well as the query
    the_nodes = ""
    for idx, item in enumerate(nodes):
        the_nodes += f"<NODE{idx+1}>\nNode ID: {item.node_id}\nText: {item.text}\n</NODE{idx+1}>\n"

    query = "<QUERY>\n{}\n</QUERY>".format(query)

    # Define the prompt template
    prompt_tmpl = PromptTemplate(
        """
    You receive a query along with a list of nodes' text and their ids. Your task is to assign a score
    to each node based on its contextual closeness to the given query. The final output is each
    node id along with its proximity score.
    Here is the list of nodes:
    {nodes_list}

    And the following is the query:
    {user_query}

    Score each of the nodes based on their text and their relevancy to the provided query.
    The score must be a decimal number between 0 and 1 so we can rank them."""
    )

    # Define an instance of GPT-4o mini and send the request
    llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})
    ordered_nodes = llm.structured_predict(
        OrderedNodes, prompt_tmpl, nodes_list=the_nodes, user_query=query
    )

    return ordered_nodes

## Define Postprocessor


The following class will use the `judger` function to rank the nodes, and filter them based on the ranks.


In [ ]:
from typing import List, Optional
from llama_index.core import QueryBundle
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.schema import NodeWithScore


class OpenaiAsJudgePostprocessor(BaseNodePostprocessor):
    def _postprocess_nodes(
        self, nodes: List[NodeWithScore], query_bundle: Optional[QueryBundle]
    ) -> List[NodeWithScore]:

        r = judger(nodes, query_bundle.query_str)

        node_ids = r.node_id
        scores = r.score

        sorted_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        num_nodes_to_select = min(3, len(sorted_scores))
        top_nodes = [sorted_scores[i][0] for i in range(num_nodes_to_select)]
        # top_nodes = [sorted_scores[i][0] for i in range(3)]

        selected_nodes_id = [node_ids[item] for item in top_nodes]

        final_nodes = []
        for item in nodes:
            if item.node_id in selected_nodes_id:
                final_nodes.append(item)

        return final_nodes

In [ ]:
judge = OpenaiAsJudgePostprocessor()

## Query Engine with Postprocessor


In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
# The `node_postprocessors` function will be applied to the retrieved nodes.
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[judge])

res = query_engine.query("Compare Retrieval Augmented Generation (RAG) and Parameter efficient Finetuning (PEFT)")

In [ ]:
print(res.response)

Summary comparison

- What they are
  - Retrieval-Augmented Generation (RAG): A hybrid system that combines an external retrieval module (indexing + search, optionally reranking) with a generative model. The retriever supplies up-to-date, relevant passages which the generator uses (via prompting) to produce responses.
  - Parameter-Efficient Fine-Tuning (PEFT): A family of fine-tuning methods that adapt a pre-trained LLM by updating only a small, efficient subset of parameters (examples include adapter-style or low-rank update methods), rather than fully re‑training the full model.

- Primary goal
  - RAG: Improve factuality, relevance and timeliness of generated outputs by augmenting generation with external knowledge at inference time.
  - PEFT: Customize or specialize an LLM for a task or domain while minimizing compute, storage, and risk of degrading pre-trained knowledge.

- Where changes happen
  - RAG: Adds or modifies the retrieval pipeline and the way retrieved context is fed 

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 8760ba85-ac25-46ee-813c-146a46fdf715
Title	 Searching for Best Practices in Retrieval-Augmented Generation:Retriever and Generator Fine-tuning
Text	 Fine-tuning within the RAG framework is crucial for optimizing both retrievers and generators. Some research focuses on fine-tuning the generator to better utilize retriever context [30–32], ensuring faithful and robust generated content. Others fine-tune the retriever to learn to retrieve beneficial passages for the generator [33–35]. Holistic approaches treat RAG as an integrated system; fine-tuning both retriever and generator together to enhance overall performance [36–38], despite increased complexity and integration challenges. Several surveys have extensively discussed current RAG systems, covering aspects like text generation [7, 8], integration with LLMs [6, 39], multimodal [40], and AI-generated content [41]. While these surveys provide comprehensive overviews of existing RAG methodologies, selecting the appropriate appr

## Optional 1: LLM Judge Scoring Visualization

After the `OpenaiAsJudgePostprocessor` runs, visualize how node scores change before and after LLM judging using a bar chart.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Run a fresh query to collect node scores before and after judging
query_text = "What is Retrieval Augmented Generation?"

# Step 1: Retrieve nodes WITHOUT the judge postprocessor (baseline scores)
retriever = index.as_retriever(similarity_top_k=10)
base_nodes = retriever.retrieve(query_text)

# Step 2: Apply the judge postprocessor to get re-ranked nodes
from llama_index.core import QueryBundle
query_bundle = QueryBundle(query_str=query_text)
judge_instance = OpenaiAsJudgePostprocessor()
judged_nodes = judge_instance.postprocess_nodes(base_nodes, query_bundle)

# Step 3: Assign judge scores (1.0 for selected, 0.0 for filtered out)
judged_ids = {n.node_id for n in judged_nodes}
before_scores = [n.score if n.score is not None else 0.0 for n in base_nodes]
after_scores = [1.0 if n.node_id in judged_ids else 0.0 for n in base_nodes]
labels = [f"Node {i+1}" for i in range(len(base_nodes))]

# Step 4: Plot
x = range(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar([i - width/2 for i in x], before_scores, width, label='Before Judging (similarity score)', color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], after_scores, width, label='After Judging (selected=1.0)', color='darkorange')

ax.set_xlabel('Retrieved Nodes')
ax.set_ylabel('Score')
ax.set_title('Node Scores Before vs After LLM Judge Re-ranking')
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.2)
plt.tight_layout()
plt.savefig('judge_scores.png', dpi=100)
print(f"Chart saved to judge_scores.png")
print(f"Nodes before judging: {len(base_nodes)} | Nodes after judging: {len(judged_nodes)}")

Chart saved to judge_scores.png
Nodes before judging: 10 | Nodes after judging: 3


## Optional 2: Compare Response Quality With vs Without LLM Judge

Run the same query through the query engine with and without the `OpenaiAsJudgePostprocessor`, then print both responses side by side to observe the difference in answer quality.

In [ ]:
compare_query = "Compare Retrieval Augmented Generation (RAG) and Parameter efficient Finetuning (PEFT)"

# Without judge postprocessor
qe_no_judge = index.as_query_engine(similarity_top_k=10)
res_no_judge = qe_no_judge.query(compare_query)

# With judge postprocessor
judge_pp = OpenaiAsJudgePostprocessor()
qe_with_judge = index.as_query_engine(similarity_top_k=10, node_postprocessors=[judge_pp])
res_with_judge = qe_with_judge.query(compare_query)

# Print both responses side by side
separator = "=" * 60
print(separator)
print("RESPONSE WITHOUT LLM JUDGE (top-k similarity only)")
print(separator)
print(res_no_judge.response)
print(f"\nSource nodes used: {len(res_no_judge.source_nodes)}")

print()
print(separator)
print("RESPONSE WITH LLM JUDGE POSTPROCESSOR")
print(separator)
print(res_with_judge.response)
print(f"\nSource nodes used: {len(res_with_judge.source_nodes)}")

RESPONSE WITHOUT LLM JUDGE (top-k similarity only)
I can summarize Retrieval-Augmented Generation (RAG) from the provided material and then compare it to Parameter-Efficient Fine-Tuning (PEFT) if you want me to bring in outside/general knowledge about PEFT. Right now, the excerpts contain detailed information about RAG but do not include descriptions of PEFT, so I can (A) describe RAG fully and (B) either wait for you to supply PEFT details or proceed to compare using general knowledge. Which would you prefer?

Summary of RAG (from the material)
- Purpose: Combines parametric memory (a pre-trained seq2seq / LLM) with non-parametric memory (an external corpus indexed as dense vectors) to improve knowledge-intensive generation and reduce hallucinations or outdated facts.
- Architecture: A neural retriever fetches relevant documents from a dense index (e.g., Wikipedia); retrieved passages are passed to a seq2seq generator/LLM which conditions on them to produce outputs. Two formulations e